In [ ]:
from typing import List, TypedDict
from langgraph.graph import StateGraph, START, END
import random

In [ ]:
class AgentState(TypedDict):
    """A TypedDict representing the state of an agent."""
    msg: List[str]

In [ ]:
def greeting_node(state: AgentState) -> AgentState:
    """A node that generates a greeting message."""
    state['msg'].append(f"Hello, {state['msg'][-1]}! How can I assist you today?")
    return state

In [ ]:
graph = StateGraph(state_schema=AgentState)
graph.add_node("greeting", greeting_node)
graph.set_entry_point("greeting")
graph.set_finish_point("greeting")

app = graph.compile()

In [ ]:
from IPython.display import display, Image
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
result = app.invoke({"msg": ["John", "Alice"]})
print(result['msg'])

In [ ]:
class AgentNewState(TypedDict):
    """A TypedDict representing the state of an agent."""
    vals: List[int]
    result: str
    name: str

def process_node(state: AgentNewState) -> AgentNewState:
    """A node that processes the input values."""
    state['result'] = f"Hi, {state['name']}! The sum of {state['vals']} is {sum(state['vals'])}."
    return state

graph = StateGraph(state_schema=AgentNewState)
graph.add_node("process", process_node)
graph.set_entry_point("process")
graph.set_finish_point("process")

app2 = graph.compile()

In [ ]:
res = app2.invoke({"vals": [57651, 52, 323], "name": "Bob"})
print(res['result'])

In [ ]:
class AgentState2(TypedDict):
    """A TypedDict representing the state of an agent."""
    name: str
    age: int
    final_message: str

In [ ]:
def first_node(state: AgentState2) -> AgentState2:
    """A first node that generates a message."""
    state['final_message'] = f"Hello, {state['name']}!"
    return state

def second_node(state: AgentState2) -> AgentState2:
    """A second node that processes the final message."""
    state['final_message'] = f"You are {state['age']} years old."
    return state

def third_node(state: AgentState2) -> AgentState2:
    """A third node that finalizes the message."""
    state['final_message'] = f"Final message: {state['name']} is {state['age']} years old."
    return state


In [ ]:
graph2 = StateGraph(state_schema=AgentState2)

graph2.add_node("first_node", first_node)
graph2.add_node("second_node", second_node)
graph2.add_node("third_node", third_node)

graph2.set_entry_point("first_node")
graph2.set_finish_point("third_node")

graph2.add_edge("first_node", "second_node")
graph2.add_edge("second_node", "third_node")

app2 = graph2.compile()

In [ ]:
from IPython.display import display, Image
display(Image(app2.get_graph().draw_mermaid_png()))

In [ ]:
res = app2.invoke({"name": "Bob", "age": 30})
print(res)

In [ ]:
class AgentState3(TypedDict):
    """A TypedDict representing the state of an agent."""
    number1: int
    number2: int
    operation: str
    answer: int
    message: str

In [ ]:
def check_route(state: AgentState3) -> AgentState3:
    """A first node that generates a message."""
    state['message'] = f"Input numbers: {state['number1']} and {state['number2']}, operation: {state['operation']}"
    return state

def decide_next_node(state: AgentState3) -> str:
    """Decides the next node based on the operation."""
    if state['operation'] == '+':
        return 'add_operation'
    elif state['operation'] == '*':
        return 'multiply_operation'
    else:
        raise ValueError(f"Unsupported operation: {state['operation']}")

def add_node(state: AgentState3) -> AgentState3:
    """A node that adds two numbers."""
    state['answer'] = state['number1'] + state['number2']
    return state

def multiply_node(state: AgentState3) -> AgentState3:
    """A node that multiplies two numbers."""
    state['answer'] = state['number1'] * state['number2']
    return state


In [ ]:
graph3 = StateGraph(state_schema=AgentState3)

graph3.add_node("check_route", check_route)
graph3.add_node("add_node", add_node)
graph3.add_node("multiply_node", multiply_node)

graph3.add_edge(START, "check_route")
graph3.add_conditional_edges(
    source="check_route",
    path=decide_next_node,
    path_map={
        'add_operation': 'add_node',
        'multiply_operation': 'multiply_node'
    }
)
graph3.add_edge("add_node", END)
graph3.add_edge("multiply_node", END)

app3 = graph3.compile()

In [ ]:
from IPython.display import display, Image
display(Image(app3.get_graph().draw_mermaid_png()))

In [ ]:
result = app3.invoke({"number1": 7567, "number2": 2000, "operation": "*"})
print(result)

In [ ]:
class AgentState4(TypedDict):
    """A TypedDict representing the state of an agent."""
    name: str
    nums: List[int]
    counter: int    

In [ ]:
def greeting_node(state: AgentState4) -> AgentState4:
    """A node that generates a greeting message."""
    state['name'] = f"Hello, {state['name']}! You have {len(state['nums'])} numbers to process."
    state['counter'] = 0
    return state

def random_number_node(state: AgentState4) -> AgentState4:
    """A node that generates a random number."""
    state['nums'].append(random.randint(1, 100))
    state['counter'] = state['counter'] + 1
    print(f"Loop: {state['counter']}, Random number: {state['nums'][-1]}")
    return state

def should_continue(state: AgentState4) -> str:
    """Decides whether to continue generating random numbers."""
    if state['counter'] < 5:
        return 'loop'
    else:
        return 'end_node'

In [ ]:
graph4 = StateGraph(state_schema=AgentState4)

graph4.add_node("greeting_node", greeting_node)
graph4.add_node("random_number_node", random_number_node)

graph4.add_edge(START, "greeting_node")
graph4.add_edge("greeting_node", "random_number_node")
graph4.add_conditional_edges(
    source="random_number_node",
    path=should_continue,
    path_map={
        'loop': 'random_number_node',
        'end_node': END
    }
)

app4 = graph4.compile()

In [ ]:
from IPython.display import display, Image
display(Image(app4.get_graph().draw_mermaid_png()))

In [ ]:
res = app4.invoke({"name": "Alice", "nums": [], "counter": -10})
print(res)